# High-dimensional image embeddings: IMSTE and UMAP

Choose dataset, graph, optimizer, and rendering settings, then compare IMSTE with UMAP on the same standardized samples. Multilevel coarsening uses IMSTE graph ranks; by default, PCA of pooled coarse-group features initializes its smallest graph. Both embeddings are displayed with `render_datashader_embedding` to keep dense plots readable.

The full gallery, including class-mixture graph analysis, is archived at `notebooks/archive/00_high_dim_mst_gallery_full.ipynb`.

This notebook does not cache downloaded datasets or fitted embeddings: every run fetches the raw MNIST variants again and refits IMSTE, gravity, and UMAP.


In [7]:
%load_ext autoreload
%autoreload 2

%config InlineBackend.figure_format = 'retina'

from pathlib import Path
import sys

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "src" / "high_dim_mst_gallery.py").is_file()),
    None,
)
if repo_root is not None:
    sys.path.insert(0, str(repo_root / "notebooks" / "src"))
    sys.path.insert(0, str(repo_root))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Select parameters and fit IMSTE

All IMSTE controls are defined in the next code cell: dataset size and seed; optimizer dimensions, epochs, batch size, learning rate, negative sampling, repulsion, and logistic loss; MST method, neighbor backend, graph sizes, component bridges, and optional kNN initialization attraction; PCA and multilevel initialization; optimizer verbosity; and every gravity second-phase setting. Change those values before running the cell to compare configurations. The UMAP and Datashader controls are grouped separately below.

With `METRIC = "cosine"`, IMSTE uses exact Prim graph construction even when `MST_METHOD = "famst"`. PCA initialization uses pooled features from the coarsest groups; disable it to use a seeded random start. Set `IMSTE_GRAVITY_ENABLED = False` to skip the second phase, or choose `"density_core"` / `"mexican_hat"` for its force mode. Gravity requires Euclidean features plus RINND, igraph, and leidenalg.

In [ ]:
from high_dim_mst_gallery import run_high_dim_mst_gallery
from IPython.display import display

# Dataset size
MAX_SAMPLES = 60_000
RANDOM_STATE = 42

# IMSTE graph and optimizer
MST_METHOD = "famst"  # or "prim" for exact graph construction
METRIC = "euclidean"  # or "cosine"; cosine uses exact Prim for IMSTE
DEVICE = "mps"  # IMSTE graph construction and optimization device
N_COMPONENTS = 2  # The comparison renderer uses the first two coordinates.
N_MSTS = 4
N_EPOCHS = 500
BATCH_SIZE = 32768
LEARNING_RATE = 0.05
NEGATIVE_RATIO = 1
LAMBDA_REP = 0.5
EPSILON = 1e-4
LOGISTIC_MARGIN = 1.0
LOGISTIC_TEMPERATURE = 0.5
IMSTE_VERBOSE = True

# FAMST candidate graph (ignored by exact Prim construction)
MST_NEIGHBOR_BACKEND = "auto"  # "auto", "rinnd", "pynndescent", or "mps_exact"
MST_NEIGHBORS = 15
MST_INTER_COMPONENT_EDGES = 5
FAMST_MAX_NEIGHBORS = 15
MST_REPRESENTATIVES_PER_COMPONENT = 10
INCLUDE_KNN_ATTRACTION = False  # Add FAMST candidate edges to the initialization graph.

# Initialization
PCA_INIT = True  # Use PCA on the coarsest graph in the multilevel initializer.
PCA_INIT_SCALING = "global"  # "global" or "none"

# Multilevel initializer (enabled by default in this notebook)
MULTILEVEL_INIT = True
MULTILEVEL_REDUCTION_RATIO = 4
MULTILEVEL_BASE_SIZE = 1024
MULTILEVEL_EPOCHS = 10

# Main refinement controls.
IMSTE_GRAVITY_ENABLED = True
IMSTE_GRAVITY_MODE = "mexican_hat"  # "mexican_hat" or "density_core"
IMSTE_GRAVITY_NEIGHBORS = 15
IMSTE_GRAVITY_RESOLUTION = 0.8
IMSTE_GRAVITY_STRENGTH = 0.1  # Set to 0 to disable gravity.
MEAN_SHIFT_NEIGHBORS = 50
MEAN_SHIFT_BANDWIDTH = 1.0
MEAN_SHIFT_STEP = 0.25
MEAN_SHIFT_ITERATIONS = 0  # Mean shift follows gravity when enabled.

gravity_settings = {
    "mode": IMSTE_GRAVITY_MODE if IMSTE_GRAVITY_ENABLED else None,
    "neighbors": IMSTE_GRAVITY_NEIGHBORS,
    "resolution": IMSTE_GRAVITY_RESOLUTION,
    "strength": IMSTE_GRAVITY_STRENGTH,
}

# UMAP comparison
UMAP_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_EPOCHS = 200
UMAP_RECURSIVE_COARSENING_RATIO = 4
REPULSION_STRENGTH = 1

# Datashader display
RENDER_WIDTH = 700
RENDER_HEIGHT = 700
RENDER_MIN_ALPHA = 40
RENDER_SPREAD_PX = 1
RENDER_BACKGROUND = "white"  # Change to "black" to compare contrast.

# Load the image datasets and fit IMSTE. Sample image grids and the helper's
# scatter plots are disabled; the comparison below uses Datashader instead.
gallery = run_high_dim_mst_gallery(
    max_samples=MAX_SAMPLES,
    n_components=N_COMPONENTS,
    n_msts=N_MSTS,
    n_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    pca_init=PCA_INIT,
    pca_init_scaling=PCA_INIT_SCALING,
    mst_method=MST_METHOD,
    mst_neighbor_backend=MST_NEIGHBOR_BACKEND,
    mst_neighbors=MST_NEIGHBORS,
    mst_inter_component_edges=MST_INTER_COMPONENT_EDGES,
    mst_representatives_per_component=MST_REPRESENTATIVES_PER_COMPONENT,
    metric=METRIC,
    include_knn_attraction=INCLUDE_KNN_ATTRACTION,
    device=DEVICE,
    verbose=IMSTE_VERBOSE,
    negative_ratio=NEGATIVE_RATIO,
    lambda_rep=LAMBDA_REP,
    epsilon=EPSILON,
    logistic_margin=LOGISTIC_MARGIN,
    logistic_temperature=LOGISTIC_TEMPERATURE,
    mst_max_neighbors=FAMST_MAX_NEIGHBORS,
    random_state=RANDOM_STATE,
    plot_sample_images=False,
    plot_embeddings=False,
    multilevel_init=MULTILEVEL_INIT,
    multilevel_reduction_ratio=MULTILEVEL_REDUCTION_RATIO,
    multilevel_base_size=MULTILEVEL_BASE_SIZE,
    multilevel_epochs=MULTILEVEL_EPOCHS,
    gravity_mode=gravity_settings["mode"],
    gravity_neighbors=gravity_settings["neighbors"],
    gravity_resolution=gravity_settings["resolution"],
    gravity_strength=gravity_settings["strength"],
    mean_shift_neighbors=MEAN_SHIFT_NEIGHBORS,
    mean_shift_bandwidth_factor=MEAN_SHIFT_BANDWIDTH,
    mean_shift_step_size=MEAN_SHIFT_STEP,
    mean_shift_iterations=MEAN_SHIFT_ITERATIONS,
)
display(gallery["summary"])
display(gallery["summary"][
    ["dataset", "gravity_mode", "gravity_neighbors", "gravity_resolution",
     "gravity_strength", "mean_shift_neighbors", "mean_shift_bandwidth_factor",
     "mean_shift_step_size", "mean_shift_iterations", "gravity_regions_packed"]
])
if gallery["load_errors"]:
    display(gallery["load_errors"])


## 2. Fit UMAP to the same samples

The gallery helper standardizes features before IMSTE. This cell applies the same standardization to each raw dataset before fitting UMAP, so both methods receive matching input rows and feature scaling. Gravity uses the same standardized input and never uses UMAP coordinates.


In [9]:
from time import perf_counter

import umap
from IPython.display import display
from sklearn.preprocessing import StandardScaler

UMAP_RESULTS = {}
UMAP_SECONDS = {}
for dataset_name, (raw_X, labels) in gallery["datasets"].items():
    X_standardized = StandardScaler().fit_transform(raw_X)
    started = perf_counter()
    embedding = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=METRIC,
        init="recursive",
        recursive_coarsening_ratio=UMAP_RECURSIVE_COARSENING_RATIO,
        compatibility_layout=False,
        nearest_neighbor_backend="rinnd",
        random_state=RANDOM_STATE,
        n_epochs=UMAP_EPOCHS,
        repulsion_strength=REPULSION_STRENGTH,
        n_jobs=1,
    ).fit_transform(X_standardized)
    UMAP_RESULTS[dataset_name] = embedding
    UMAP_SECONDS[dataset_name] = perf_counter() - started
    print(f"{dataset_name}: UMAP fit in {UMAP_SECONDS[dataset_name] / 60:.1f} min")

comparison = gallery["summary"][["dataset", "samples", "input_features", "seconds"]].rename(
    columns={"seconds": "IMSTE seconds"}
)
comparison["UMAP seconds"] = comparison["dataset"].map(UMAP_SECONDS)
display(comparison)


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

MNIST (28×28): UMAP fit in 0.3 min


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

Fashion-MNIST (28×28): UMAP fit in 0.3 min


/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:378: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/f.costa/.venvs/py312/lib/python3.12/site-packages/sklearn/utils/extmath.py:379: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users

Kuzushiji-MNIST (28×28): UMAP fit in 0.3 min


,dataset,samples,input_features,IMSTE seconds,UMAP seconds
0,MNIST (28×28),60000,784,24.975515,18.648739
1,Fashion-MNIST (28×28),60000,784,23.209656,19.114296
2,Kuzushiji-MNIST (28×28),60000,784,24.968824,19.611794


## 3. Compare the embeddings with Datashader

Each panel rasterizes all points at the selected image size. Adjust the rendering settings above to change the mark spread, opacity, resolution, or background.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from imste.visualization import render_datashader_embedding

DATASHADER_IMAGES = {}
summary = gallery["summary"].set_index("dataset")
IMSTE_SECONDS = summary["imste_seconds"].to_dict()
GRAVITY_SECONDS = summary["gravity_seconds"].to_dict()
MEAN_SHIFT_SECONDS = summary["mean_shift_seconds"].to_dict()
s = 7
for dataset_name, (_, labels) in gallery["datasets"].items():
    embeddings = {
        ("IMSTE · multilevel start" if MULTILEVEL_INIT else "IMSTE · PCA start"): gallery["base_embeddings"][dataset_name],
    }
    if IMSTE_GRAVITY_ENABLED or MEAN_SHIFT_ITERATIONS > 0:
        embeddings["IMSTE · gravity → mean shift"] = gallery["embeddings"][dataset_name]
    embeddings["UMAP"] = UMAP_RESULTS[dataset_name]
    fig, axes = plt.subplots(1, len(embeddings), figsize=(9 * len(embeddings), 7), constrained_layout=True, squeeze=False)
    for ax, (method, embedding) in zip(axes.ravel(), embeddings.items()):
        image = render_datashader_embedding(
            embedding,
            labels,
            width=300*s,
            height=250*s,
            min_alpha=RENDER_MIN_ALPHA,
            background=RENDER_BACKGROUND,
            spread_px=3,
        )
        DATASHADER_IMAGES[(dataset_name, method)] = image
        ax.imshow(image.to_pil())
        elapsed = (
            IMSTE_SECONDS[dataset_name] + GRAVITY_SECONDS[dataset_name] + MEAN_SHIFT_SECONDS[dataset_name]
            if method == "IMSTE · gravity → mean shift"
            else IMSTE_SECONDS[dataset_name]
            if method.startswith("IMSTE")
            else UMAP_SECONDS[dataset_name]
        )
        runtime = f"{elapsed / 60:.1f} min" if elapsed >= 60 else f"{elapsed:.1f} s"
        ax.set_title(f"{dataset_name} — {method} ({runtime})")
        ax.set_axis_off()
    display(fig)
    plt.close(fig)
